# 多変数のホテリング理論による異常検知

4章で解説した1変数のホテリング理論による異常検知は、解析計算に基づく理論の明快さや統計ライブラリによる実装のしやすさから、基礎的な異常検知アルゴリズムの一つとして広く利用されています。

一方で実際の課題に適用しようとすると、1変数のみでは正常と異常を十分に区別できないケースが頻繁に発生します。
例えばクレジットカードの不正利用検知では、高額購入という情報だけを見ると異常に思えても、旅行先での連続利用であれば正常と考えられる場合があります。一方で、少額でも離れた地域で同時に決済が行われれば、不正の可能性が高いと判断できます。この例では金額という1変数だけでは不十分で、場所や時間といった他の変数を組み合わせてモデル化することで、より高い検知性能が得られます。

ここでは多変数のホテリング理論による異常検知を、Pythonを用いて以下の手順で実装する方法を解説します。

- A. モデルの学習
- B. 推論

なおデータセットには2章で作成したサンプルデータを使用し、入力する変数は`temp1`と`temp2`の2変数を、ターゲットとする誤報率としては0.0027を採用します。

## A. モデルの学習

学習フェーズでは、最尤推定による多次元正規分布パラメータの推定、および異常度のしきい値算出を実施します。

多変数のホテリング理論では、**サンプルサイズ**$N$と**変数の数**$M$の関係に応じて以下のように異常度$a(x) = (\boldsymbol{x}-\hat{\boldsymbol{\mu}})^\top \hat{\boldsymbol{\Sigma}}^{-1} (\boldsymbol{x}-\hat{\boldsymbol{\mu}})$が従う確率分布を使い分けます（$\hat{\boldsymbol{\mu}}$は標本平均ベクトル、$\hat{\boldsymbol{\Sigma}}$は標本分散共分散行列）。

- $N \gg M$のとき（$N$が十分大きく、$M$が極端に大きくない）：異常度の分布は自由度$M$の**カイ二乗分布**に近似できる
- $N \gg M$でないとき：異常度を$\frac{N-M}{(N+1)M}$倍した統計量は自由度$(M,N − M)$の**F分布**に従う

ここではそれぞれの場合について、Pythonでモデルの学習を実装する方法を解説します。

### $N \gg M$が成り立つ場合の学習の実装

$N \gg M$が成り立つ場合、異常度$a(x) = (\boldsymbol{x}-\hat{\boldsymbol{\mu}})^\top \hat{\boldsymbol{\Sigma}}^{-1} (\boldsymbol{x}-\hat{\boldsymbol{\mu}})$が自由度$M$のカイ二乗分布に従うことを前提に、以下のように多次元正規分布（正常のモデル）のパラメータ$\hat{\boldsymbol{\mu}}$（コード中の`mu`）、$\hat{\boldsymbol{\Sigma}}$（`Sigma`）および異常度のしきい値$a_{th}$（`a_th`）を求めます。

In [ ]:
# コード6.1 多変数のホテリング理論による異常検知の実装例（学習）
import pandas as pd
import numpy as np
from scipy import stats

###### 学習データの読み込みと前処理######
# CSVからPandas DataFrameにデータ読み込み
df = pd.read_csv("./datasets/ch2_dataset_train.csv")
# "temp2","temp1"変数に欠測があるデータを削除
df_dropna = df.dropna(subset=["temp2", "temp1"])
# 正常データのみを抽出
df_normal = df_dropna[df_dropna["label"] == "normal"]
# "temp2","temp1"列のみ取り出してNumpyのndarray化し、学習データとする
X_train = df_normal[["temp2", "temp1"]].to_numpy()

###### 学習ステップ1. 正常のモデルを作成する######
mu = np.mean(X_train, axis=0) # 標本平均ベクトルμを算出
# 標本分散共分散行列Σを算出（転置と⾃由度ddofに注意）
Sigma = np.cov(X_train.T, ddof=0)

###### 学習ステップ2. 異常を表す指標（異常度）を定義する######
# 式を定義するのみでプログラム上は処理を実施しない

###### 学習ステップ3. 異常度にしきい値を設ける######
TARGET_FP_RATE = 0.0027 # ターゲットとする誤報率（正規分布の3σ相当=0.0027）
n_features = X_train.shape[1] # 変数の数M
# ⾃由度（M）のカイ二乗分布の累積分布関数の逆関数からしきい値を算出
a_th = stats.chi2.ppf(1-TARGET_FP_RATE, df=n_features)

###### 学習で求めたパラメータをすべて表示######
print(f"mu={mu}")
print(f"Sigma={Sigma}")
print(f"a_th={a_th}")

### $N \gg M$が成り立たない場合の学習の実装

$N \gg M$が成り立たない場合、異常度$a(x) = (\boldsymbol{x}-\hat{\boldsymbol{\mu}})^\top \hat{\boldsymbol{\Sigma}}^{-1} (\boldsymbol{x}-\hat{\boldsymbol{\mu}})$異常度を$\frac{N-M}{(N+1)M}$倍した統計量が自由度$(M,N − M)$の**F分布**に従うことを前提に、以下のように多次元正規分布（正常のモデル）のパラメータ$\hat{\boldsymbol{\mu}}$（コード中の`mu`）、$\hat{\boldsymbol{\Sigma}}$（`Sigma`）および異常度のしきい値$a_{th}$（`a_th`）を求めます。

In [ ]:
###### 学習ステップ1. 正常のモデルを作成する######
mu = np.mean(X_train, axis=0) # 標本平均ベクトルμを算出
# 標本分散共分散行列Σを算出（転置と⾃由度ddofに注意）
Sigma = np.cov(X_train.T, ddof=0)

###### 学習ステップ2. 異常を表す指標（異常度）を定義する######
# 式を定義するのみでプログラム上は処理を実施しない

###### 学習ステップ3. 異常度にしきい値を設ける######
TARGET_FP_RATE = 0.0027 # ターゲットとする誤報率（正規分布の3σ相当=0.0027）
sample_size = len(X_train) # サンプルサイズN
n_features = X_train.shape[1] # 変数の数M
# ⾃由度（M, N-M）のF分布の累積分布関数の逆関数からしきい値を算出
a_th = (sample_size+1)*n_features/(sample_size-n_features) \
* stats.f.ppf(1-TARGET_FP_RATE, dfn=n_features, dfd=sample_size-n_features)

###### 学習で求めたパラメータをすべて表示######
print(f"mu={mu}")
print(f"Sigma={Sigma}")
print(f"a_th={a_th}")

カイ二乗分布を使用した場合と比べて異常度のしきい値がやや大きくなっており、誤報率を減らす方向（見逃し寄り）にしきい値が設定されることが分かります。

求めたパラメータ`mu`（標本平均ベクトル$\hat{\boldsymbol{\mu}}$）、`Sigma`（標本分散共分散行列$\hat{\boldsymbol{\Sigma}}$）および異常度のしきい値`a_th`（$a_{th}$）は設定ファイルなどに保存して、推論フェーズで再利用します（ここでは簡単のため、得られたパラメータを推論時に直接コードに記述することとします）。

## C. 推論

学習フェーズで求めたパラメータとしきい値を用いて、推論データに対する異常度の算出と異常判定を行います。なお、異常度の分布にカイ二乗分布を用いる（$N \gg M$が成り立つ）場合もF分布を用いる（$N \gg M$が成り立たない）場合も、推論の実装方法に変化はありません。

In [ ]:
# コード6.2 多変数のホテリング理論による異常検知の実装例（推論）

###### 学習したパラメータをここに記載
MU = [350.23771153, 400.21552883] # 標本平均ベクトルμ
SIGMA = [[26.04088919, 14.52874949],
         [14.52874949, 24.58401158]] # 標本分散共分散行列Σ
A_TH=11.82900701194368 # 異常度のしきい値

###### 推論データの読み込みと前処理######
# CSVからPandas DataFrameにデータ読み込み
df_inference = pd.read_csv('./datasets/ch2_dataset_inference.csv')
# "temp2","temp1"変数に欠測があるデータを削除
df_inference_dropna = df_inference.dropna(subset=["temp2", "temp1"])
# "temp2","temp1"列のみ取り出してNumpyのndarray化し、推論データとする
X_inference = df_inference_dropna[["temp2", "temp1"]].to_numpy()
# 標本分散共分散行列の逆行列
Sigma_inv = np.linalg.inv(np.array(SIGMA))

###### 推論を実行######
# 異常度を算出
X_dev = (X_inference-np.array(MU)).T # 推論データと平均ベクトルとの差（x-μ）
Dev_Sigma = X_dev.T @ Sigma_inv # (x-μ)^T * Σ^-1
anomaly_scores = np.sum(np.multiply(Dev_Sigma, X_dev.T), axis=1)
# しきい値により異常の有無を判定
pred = np.where(anomaly_scores > A_TH, 'anomaly', 'normal')
# 推論結果を表示
print(pred)

推論結果の決定境界（異常と正常の判定の境界）を可視化してみます。

In [ ]:
# 推論結果の決定境界を可視化
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

# 異常度と異常判定をDataFrameに列として追加
df_inference_dropna['anomaly_score'] = anomaly_scores
df_inference_dropna['prediction'] = pred
# 描画用のFigureとAxesを生成
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 4))

###### 正常と異常の範囲を色分け ######
# (x1,x2)格子点を作成（'temp2'をx1としていることに注意）
x1_grid = np.linspace(df_inference_dropna['temp2'].min()-5,
                      df_inference_dropna['temp2'].max()+5, 200)
x2_grid = np.linspace(df_inference_dropna['temp1'].min()-5,
                      df_inference_dropna['temp1'].max()+5, 200)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
X_grid = np.c_[X1.ravel(), X2.ravel()]
# 異常度を算出（式6.53に従う）
X_dev_grid = (X_grid-np.array(MU)).T  # 推論データと平均ベクトルとの差（x-μ）
Dev_Sigma_grid = X_dev_grid.T @ Sigma_inv  # (x-μ)^T * Σ^-1
anomaly_scores_grid = np.sum(np.multiply(Dev_Sigma_grid, X_dev_grid.T), axis=1)
# しきい値判定
pred_grid = np.where(anomaly_scores_grid > A_TH, 0, 1)
# 正常と異常の境界をプロット
pred_pivot = pred_grid.reshape(X1.shape)
ax.contourf(X1, X2, pred_pivot,
            cmap=cm.gray, alpha=0.5)

###### 各データを散布図としてプロット ######
sns.scatterplot(data=df_inference_dropna, x='temp2', y='temp1',
    hue='label', palette=['#999999', '#111111'],
    ax=ax
)
# 凡例を追加
ax.legend()
# グラフを表示
plt.show()

背景の塗りつぶし色がしきい値判定の結果を表しており、暗い部分が異常判定を表します。また散布図が推論に使用したデータ点を表しており、マーカー色（凡例）が正解ラベルを示します。

本手法では正常分布からの統計的距離に基づき異常判定を行うため、正常データから離れた任意の方向の異常を検知できることがわかります。したがって、教師あり学習では検出困難な学習データに含まれていない未知カテゴリの異常であっても、正常分布から外れている限り検出可能であるという利点があります。